# Lab 6: Randomized Response

***
- **FIRST name**: Moffat
- **LAST name**: Muriithi
- **Student ID**: 1864875

Leave blank if individual:
- **Collaborator names**:
- **Collaborator student IDs**:
***

In today's lab, you will learn:

1. apply randomized response mechanisms;
2. obtain the estimated count of true responses;
3. apply biased randomized response mechanisms;
4. calculate the relative error.

For this lab, you'll need the dataset `adult.csv`.

### Instructions

- **Collaboration**: You must submit your own work. The collaboration policy for the labs is Consultation Collaboration. You may verbally discuss concepts with your classmates, without exchanging written text, code, or detailed advice. You must develop your own solution and submit your own work. All sources of information used including books, websites, students you talked to, must be cited in the submission. Please see the course FAQ document for details on this collaboration policy. We will adhere to current Faculty of Science guidelines on dealing with suspected cases of plagiarism.
- **Software**: We highly recommend that students use Syzygy for completing labs and assignments. This is the software used by the TAs in the course, and we can guarantee that there will be no issues with incompatible environments or imports.
- **Filling out the Notebook**: You must use this notebook to complete your lab. You will execute the questions in the notebook. The questions might ask for a short answer in text form or for you to write and execute a piece of code. Make sure you enter your answer in either case only in the cell provided.
- **Important**:  Do not use a different cell, do not delete cells, and do not create a new cell. Creating new cells for your code is not compatible with the auto-grading system we are using and thus your assignment will not get grading properly and you will lose marks for that question. As a reminder you must remove the raise NotImplementedError() statements from each question when answering.
- **Rules for Datasets**: Any datasets used in the lab cannot be imported from cloud storage, e.g google drive, and must be read from a file either on your local computer or uploaded to the google colab notebook. Importing from cloud storage will result in a zero.
- **Submission Formatting**: When you are done, you will submit your work from the notebook. Make sure to save your notebook before running it, and then submit on Canvas the notebook file with your work completed. Name your file with an L followed by the lab number (ex: L6.ipynb). Failure to do so will result in a zero! Finally your name must be written at the top of the lab or assignment document.


In [1]:
# Don't change this cell; just run it.
import numpy as np
from numpy.random import default_rng
rng = default_rng()
import pandas as pd
from scipy.optimize import minimize

# These lines do some fancy plotting magic.
import matplotlib
# This is a magic function that renders the figure in the notebook, instead of displaying a dump of the figure object.
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

## Importing and Cleaning Data


### Question 1
Load the dataset `adult.csv` into a pandas dataframe called `df`. Drop all rows with missing values and store the result back into `df`.

In [27]:
# YOUR CODE HERE
data_im = pd.read_csv('adult.csv')
df = pd.DataFrame(data_im)
#As rows with missing values are represented by a question,
#we will use the replace method to replace them with nan
df = df.replace('?', np.nan)
df = df.dropna()

df

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K


In [ ]:
#TEST CELL: do not delete!

Now convert the values in the income column to 1 (>50K) and 0 (<=50K)

In [28]:
# YOUR CODE HERE
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(["<=50K",">50K"])

target_transform = le.transform(df["income"])
df["income"] = target_transform
df["income"]

1        0
3        0
4        0
5        0
6        0
        ..
32556    0
32557    0
32558    1
32559    0
32560    0
Name: income, Length: 30162, dtype: int64

In [ ]:
#TEST CELL: do not delete!

## Randomized Response

Now let's implement a Randomized Response mechanism.  Let's assume the query is about the income being <=50K or >50K.  The value in the `income` column is the true answer.

### Question 2
Consider the following randomized response to the query on each row of whether the income is <=50K or >50K.  The reported answer is as follows:

Flip a fair coin.
- If it's heads, respond truthfully:  the reported answer is the same as the value in `income`.
- If it's tails, flip a fair coin again:
   - If it's heads, the reported answer has a value of 1.
   - If it's tails, the reported answer has a value of 0.
  
Implement a function for this mechanism in the code cell below.
The function should accept a value `0` or `1` and return as output the reported answer according to the randomized response above.

In [29]:
#adding noise to the data
def rand_resp_c1(x):
    random_choice = np.random.choice(['heads', 'tails'])
    if random_choice == 'heads':
        return x
    else:
        random_choice2 = np.random.choice(['heads', 'tails'])
        if random_choice2 == 'heads':
            return 1
        else:
            return 0
df['income_rrc1'] = [rand_resp_c1(x) for x in df['income']]
df['income_rrc1']

1        0
3        0
4        0
5        0
6        1
        ..
32556    0
32557    1
32558    1
32559    1
32560    0
Name: income_rrc1, Length: 30162, dtype: int64

In [ ]:
#TEST CELL: do not delete!

### Question 3
Now get the **estimate** for the true number of people with income >50K.  Write this result into the variable `est_yes`.  Given the number of reported `1`'s (income >50K), we know how to estimate the proportion or number of true `1`'s.

Calculate the true number of people with income >50K (the true answer in the data we imported) and write it into the variable `true_yes`.

**Hint**: recall the derivations we did in class to analyse our randomized response:
- $m$ is the sum of true answers
- $n$ is the total number of participants
- $z$ is the sum of observed obfuscated answers
- $xi$  is the true value corresponding to an individual i
- $yi$ is the published (obfuscated) value corresponding to an individual i.

We also know that we can only estimate $m$, and we define that as $\tilde{m}$. This meanse we can only observe that $z = \sum_{i} y_i $

Following the derivations of $\mathbb{E} [ { y_i } ] = \frac{3}{4}x_i + \frac{1}{4}(1 - x_i)$ we can isolate a formula for $\tilde{m}$ that uses the variables $z$ and $n$.

In [34]:
# YOUR CODE HERE
#z being the number of true positives and false positives
#n is the length of the table
#use m for the sum of true yes'
z = 0
for x in df['income_rrc1']:
    if x == 1:
        z += 1
m = 0
for y in df['income']:
    if y == 1:
        m += 1
        
est_yes = 2 * (z - (len(df)/4))
true_yes = m

In [ ]:
#TEST CELL: do not delete!

Now consider a more general randomized response mechanism to the query on each row of whether the income is <=50K or >50K. This more general mechanism, that takes a parameter `bias` as an argument, is as follows:


With probability `1/2 + bias`, respond truthfully.
With probability `1/2 - bias`, respond with the opposite of the true answer.

Implement a function for this mechanism.


In [35]:
def rand_resp_cg(x,b):

    # YOUR CODE HERE
    #Calculate the prob of telling the truth
    p_truth = 0.5 + b

    #Then simulate a weighted coin flip
    rand_flip = np.random.rand()

    #Apply the logic
    if rand_flip < p_truth:
        return x
    else:
        return 1 - x

In [36]:
#TEST CELL: do not delete!

Now as we did for the coin flip case, we want to compare the estimated number of people with income >50K (the responses from the randomized response) with the true number (from the dataset - the `income` column).  The difference between the two values is the error in estimation.

For this generalized randomized response function, we have a parameter, `b`, which represents the bias and is part of the probability of responding truthfully or not. Thus, it functions as a privacy parameter. We want to analyze the error in the estimate of the number of people with income >$50K with respect to this privacy parameter, `b`.

Therefore, let's first set up an array, which we will call `bias_values`, to hold various values of `b` that we will test. We will test `b` values from 0.05 to 0.45 in increments of 0.05.

In [37]:
# YOUR CODE HERE
bias_values = np.arange(0.05, 0.50, 0.05)
bias_values

array([0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45])

In [ ]:
#TEST CELL: do not delete!

### Question 4

You are now going to simulate randomized responses using different values of the bias parameter.
You are provided with a list of bias values in the array `bias_values`.

For each bias value `b`, apply your `rand_resp_cg(x, b)` function to the `income` column `x`.  
Store the result in a 2D array called `rrg`, where each **row** corresponds to a different **bias value** and each **column** corresponds to an individual’s randomized response.

**Structure of `rrg`:**

- `rrg[i][j]` is the randomized response for the `j`‑th individual when using a bias of `bias_values[i]`.
- The shape of the array should be:
```python
rrg.shape == (len(bias_values), len(x))
```
After the first two lines of code in the cell below, write your code to fill `rrg` with the randomized responses and `est_yes` will contain the *estimated* number of true `1` responses for each value of bias.  So `est_yes[0]` will contain the estimated number of true yes responses when the bias is 0.05.


**Hint** : this is similar to calculating `est_yes` in Question 3, however now you have to consider how to include the bias in your derivation.

In [39]:
rrg=np.zeros((len(bias_values),len(df['income'])))
est_yes=np.zeros(len(bias_values))

# YOUR CODE HERE
x = df['income'].values
n = len(x)

# Loop through each bias value
for i, b in enumerate(bias_values):
    rrg[i] = [rand_resp_cg(val, b) for val in x]
    z = np.sum(rrg[i])
    est_yes[i] = (z - n * (0.5 - b)) / (2 * b)

In [40]:
#TEST CELL: do not delete!

Implement a function called `rel_error` to calculate the relative error between the true number of `1`s (in the `income` column) and the estimated number (relative to the true number).  The function should accept arrays as input and output an array of relative errors.  (So it should be able to accept arrays of numbers and output an array of the relative errors.)

In [41]:
def rel_error(true,estim):
    # YOUR CODE HERE
    return np.abs(true - estim) / true

print(rel_error(np.sum(rrg,1),est_yes))

[0.35674893 0.43005067 0.4091653  0.37341615 0.31608343 0.29932104
 0.22826655 0.16724768 0.09219715]


In [ ]:
#TEST CELL: do not delete!

# Rubric

| Question | Points|
|----------|----------|
| 1.   | 10   |
| 2.    | 10   |
| 3.    | 25   |
| 4.   | 25   |
| Total:    | 70   |
